# 第一阶段：项目启动与初步数据探索

本 Notebook 用于完成信贷违约预测项目的第一阶段工作。

主要目标：

- 加载本地数据集
- 识别可能的目标变量
- 检查缺失值、重复值和基础数据质量
- 为下一阶段的数据清洗和EDA做准备


## 1. 导入工具库与数据集

In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

DATA_PATH = Path("credit_data.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"找不到数据文件： {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
df.head()

## 2. 数据基础概览

2.1 数据形状

In [ ]:
print(f"row: {df.shape[0]:,}")
print(f"column {df.shape[1]:,}")

row 800,000
column: 47


2.2 基础信息

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 800000 entries, 0 to 799999
Data columns (total 47 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   id                  800000 non-null  int64  
 1   loanAmnt            800000 non-null  float64
 2   term                800000 non-null  int64  
 3   interestRate        800000 non-null  float64
 4   installment         800000 non-null  float64
 5   grade               800000 non-null  str    
 6   subGrade            800000 non-null  str    
 7   employmentTitle     799999 non-null  float64
 8   employmentLength    753201 non-null  str    
 9   homeOwnership       800000 non-null  int64  
 10  annualIncome        800000 non-null  float64
 11  verificationStatus  800000 non-null  int64  
 12  issueDate           800000 non-null  str    
 13  isDefault           800000 non-null  int64  
 14  purpose             800000 non-null  int64  
 15  postCode            799999 non-null  float64


2.3 基础统计

In [5]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,800000.0000,NaN,NaN,NaN,399999.5000,230940.2520,0.0000,199999.7500,399999.5000,599999.2500,799999.0000
loanAmnt,800000.0000,NaN,NaN,NaN,14416.8189,8716.0862,500.0000,8000.0000,12000.0000,20000.0000,40000.0000
term,800000.0000,NaN,NaN,NaN,3.4827,0.8558,3.0000,3.0000,3.0000,3.0000,5.0000
interestRate,800000.0000,NaN,NaN,NaN,13.2384,4.7658,5.3100,9.7500,12.7400,15.9900,30.9900
installment,800000.0000,NaN,NaN,NaN,437.9477,261.4604,15.6900,248.4500,375.1350,580.7100,1715.4200
grade,800000,7,B,233690,NaN,NaN,NaN,NaN,NaN,NaN,NaN
subGrade,800000,35,C1,50763,NaN,NaN,NaN,NaN,NaN,NaN,NaN
employmentTitle,799999.0000,NaN,NaN,NaN,72005.3517,106585.6402,0.0000,427.0000,7755.0000,117663.5000,378351.0000
employmentLength,753201,11,10+ years,262753,NaN,NaN,NaN,NaN,NaN,NaN,NaN
homeOwnership,800000.0000,NaN,NaN,NaN,0.6142,0.6757,0.0000,0.0000,1.0000,1.0000,5.0000


2.4 缺失值统计

In [25]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_ratio": df.isna().mean()
}).sort_values("missing_count", ascending=False)

missing_summary.head(20)

,missing_count,missing_ratio
n11,69752,0.0872
employmentLength,46799,0.0585
n8,40271,0.0503
n6,40270,0.0503
n12,40270,0.0503
n9,40270,0.0503
n13,40270,0.0503
n14,40270,0.0503
n5,40270,0.0503
n0,40270,0.0503


2.5 区分数值字段和类别字段

In [6]:
numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = df.select_dtypes(exclude=["number"]).columns.tolist()

print(f"数值字段数量： {len(numeric_cols)}")
print(numeric_cols)

print(f"\n类别/非数值字段数量： {len(categorical_cols)}")
print(categorical_cols)

数值字段数量： 42
['id', 'loanAmnt', 'term', 'interestRate', 'installment', 'employmentTitle', 'homeOwnership', 'annualIncome', 'verificationStatus', 'isDefault', 'purpose', 'postCode', 'regionCode', 'dti', 'delinquency_2years', 'ficoRangeLow', 'ficoRangeHigh', 'openAcc', 'pubRec', 'pubRecBankruptcies', 'revolBal', 'revolUtil', 'totalAcc', 'initialListStatus', 'applicationType', 'title', 'policyCode', 'n0', 'n1', 'n2', 'n3', 'n4', 'n5', 'n6', 'n7', 'n8', 'n9', 'n10', 'n11', 'n12', 'n13', 'n14']

类别/非数值字段数量： 5
['grade', 'subGrade', 'employmentLength', 'issueDate', 'earliesCreditLine']


2.6 目标变量检查

本项目的主要目的是预测用户的违约概率，基于本目的推测 “isDefault” 可以作为目标变量.

In [26]:
TARGET = "isDefault"

if TARGET not in df.columns:
    raise ValueError(f"目标字段 '{TARGET}' 没有在数据集中找到。")

target_counts = df[TARGET].value_counts(dropna=False)
target_ratio = df[TARGET].value_counts(normalize=True, dropna=False)

target_summary = pd.DataFrame({
    "count": target_counts,
    "ratio": target_ratio
})

target_summary

,count,ratio
isDefault,,
0,640390,0.8005
1,159610,0.1995


2.7 重复值检查

In [ ]:
duplicate_rows = df.duplicated().sum()
print(f"重复行数量： {duplicate_rows:,}")

if "id" in df.columns:
    print(f"唯一 ID 数量： {df['id'].nunique():,}")
    print(f"总行数： {len(df):,}")
    print(f"id 是否唯一： {df['id'].is_unique}")